In [ ]:
import os
import sys
from pathlib import Path
sys.path.append(os.path.join(Path().resolve(), '..'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from cycler import cycler
from matplotlib.animation import FuncAnimation
from typing import List, Tuple

from data import data_loader
from src.module.dmd import DMD
from src.module.regime import Regime
from src.utils.tools4preprocess import create_variables

plt.rcParams['text.color'] = 'white'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 15
plt.rcParams['axes.titlecolor'] = 'white'
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.width'] = 1.0
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.width'] = 1.0
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.linewidth'] = 1.0
plt.rcParams["axes.facecolor"] = "#191919"
plt.rcParams["axes.edgecolor"] = "white"
plt.rcParams["figure.facecolor"] = "#191919"
plt.rcParams["figure.edgecolor"] = "white"
plt.rcParams["legend.facecolor"] ="dimgray"
plt.rcParams["legend.labelcolor"] ="white"
plt.rcParams['axes.prop_cycle'] = cycler('color', ['#8dd3c7', '#feffb3', '#bfbbd9', '#fa8174', '#81b1d2', '#fdb462', '#b3de69', '#bc82bd', '#ccebc4', '#ffed6f'])

## Synthetic Data

|             |     Regime1    |     Regime2    |
|  ---        |  ----          |  ----          |
|  Interval1  |    (1, 1000)   |  (1001, 2000)  |
|  Interval2  |  (2001, 3000)  |  (3001, 4000)  |

## 1. Initialize

prepare initial k regimes (in this case, k = 2)

In [ ]:
TRAIN = 500
THRESH = 0.99
h = 32
k = 2

In [ ]:
def regime_initializer(data: np.ndarray, k: int, init_window: int, st_points: tuple, h: int, thresh: float) -> Tuple[list[Regime], list[float]]:
    cand_models = list()
    preds = list()
    for i in range(k):
        X, Y = create_variables(data[:, st_points[i] : st_points[i] + init_window], h=h)
        d, n = X.shape
        model = DMD(d=d, h=h, thresh=THRESH)
        model.fit(X, Y)
        preds.append(model.predict(X[:, 0], interval=(0, 1000), show=False))
        cand_models.append(Regime(model, []))
    return preds, cand_models

In [ ]:
data = data_loader("").T
X, Y = create_variables(data, h=h)
d, n = X.shape

plt.figure(figsize=(20, 2.5))
plt.plot(data.T)
plt.show()

In [ ]:
preds, cand_models = regime_initializer(data=data, k=k, init_window=TRAIN, st_points=(0, 1000), h=h, thresh=THRESH)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))

for i in range(k):
    ax[i].plot(preds[i].T)
    ax[i].set_title(f'Regime{i + 1}')
    ax[i].set_xlabel('Time')
    ax[i].set_ylim(-2, 8.5)

fig.suptitle("Initial Models")
fig.tight_layout()
plt.show()

## Optimization

In [ ]:
def RMSE(X: np.ndarray, Y: np.ndarray) -> float:
    _err = 0
    for i in range(X.shape[0]):
        _err += np.sqrt(np.mean((X[i, :] - Y[i, :]) ** 2))
    return _err

def record_intervals(models: list, T: int, STEP: int, WINDOW: int) -> None:
    for model in models:
        model.intervals = []
    for i, t in enumerate(range(0, T, STEP)):
        models[optimal_path[i]].update_intervals(t + WINDOW - STEP, t + WINDOW)

def path2interval(models: List[List[int]], names: List[str]) -> List[Tuple[int, int, str]]:
    intervals = [(label, st, en) for model, label in zip(models, names) for st, en in model]
    intervals = sorted(intervals, key=lambda x: x[1])

    redistributed = []
    while len(intervals) > 1:
        (label1, st1, en1), (label2, st2, en2) = intervals[0], intervals[1]
        if st2 < en1:
            mid = (st2 + en1) // 2
            redistributed.append((label1, st1, mid))
            intervals = [(label2, mid, en2)] + intervals[2:]
        else:
            redistributed.append((label1, st1, en1))
            intervals = intervals[1:]
    redistributed.append(intervals[0])

    return redistributed

### E-step

In [ ]:
SWITCH_COST = 1.5
STEP = 1
WINDOW = 200

In [ ]:
def estimator(k: int, cand_models: list, X: np.ndarray, STEP: int, h: int, WINDOW: int, switch_cost: float) -> list:
    prev_costs = [0] * k
    curr_costs = [0] * k
    prev_paths = [[] for _ in range(k)]
    curr_paths = [[] for _ in range(k)]

    d, n = X.shape
    T = n - WINDOW
    for t in range(0, T, STEP):
        for j in range(len(cand_models)):
            min_idx = np.argmin(prev_costs)
            sub_X = X[[s * h for s in range(d // h)], t : t + WINDOW]
            pred = cand_models[j].model.predict(X[:, t], interval=(0, WINDOW))
            ll = RMSE(sub_X, pred)
            if prev_costs[min_idx] + switch_cost > prev_costs[j]:
                curr_costs[j] = prev_costs[j] + ll
                curr_paths[j] = prev_paths[j] + [j]
            else:
                curr_costs[j] = prev_costs[min_idx] + ll + switch_cost
                curr_paths[j] = prev_paths[min_idx] + [j]
        prev_costs = curr_costs[:]
        prev_paths = [path[:] for path in curr_paths]

    for i in range(k):
        cand_models[i].intervals = []

    optimal_path = curr_paths[np.argmin(prev_costs)]
    for i, t in enumerate(range(0, T, STEP)):
        st = t + WINDOW + h
        en = t + WINDOW + h + STEP
        cand_models[optimal_path[i]].update_intervals(st, en)

    return cand_models

In [ ]:
cand_models = estimator(k=k, cand_models=cand_models, X=X, STEP=STEP, h=h, WINDOW=WINDOW, switch_cost=SWITCH_COST)

In [ ]:
total_assignment = [[] for _ in range(k)]
for i in range(k):
    total_assignment[i] = cand_models[i].intervals

plt.figure(figsize=(20, 4))
plt.axvspan(0, WINDOW + h, color='#fa8174')
COLORS = [i["color"] for i in plt.rcParams["axes.prop_cycle"]]
for interval, color in zip(total_assignment, COLORS):
    for st, en in interval:
        plt.axvspan(st, en, color=color)
plt.plot(data.T, color="black", linestyle="dashed")
print(total_assignment)

### M-step

In [ ]:
cand_models = list()
for intervals in total_assignment:
    for i, (st, en) in enumerate(intervals):
        if i == 0:
            model = DMD(d=d, h=h, thresh=THRESH, dt=0.01)
            model.fit(X[:, st - h : en - h], Y[:, st - h : en - h])
        else:
            for xt, yt in zip(X[:, st - h : en - h].T, Y[:, st - h : en - h].T):
                model.learn_one(xt.reshape(-1, 1), yt.reshape(-1, 1))
    cand_models.append(Regime(model, []))

## Mode

In [ ]:
preds = [[] for _ in range(k)]
t0s = [[] for _ in range(k)]
for i in range(k):
    t0s[i] = total_assignment[i][0][0] - h
    preds[i] = cand_models[i].model.predict(X[:, t0s[i]], interval=(0, 500), show=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(np.arange(t0s[0], t0s[0] + 500), preds[0].T)
ax[0].set_title('Regime1')
ax[0].set_xlabel('Time')

ax[1].plot(np.arange(t0s[1], t0s[1] + 500), preds[1].T)
ax[1].set_title('Regime2')
ax[1].set_xlabel('Time')

plt.show()

## GridSearch